# Ring Modulator: Optical Impulse Response

This notebook simulates the transient optical response of a ring resonator
modulator using **Temporal Coupled-Mode Theory (t-CMT)**. An optical pulse is
launched into a bus waveguide coupled to the ring. We observe how the ring
energy $a(t)$ builds up and decays, and how this shapes the transmitted field.

## t-CMT equations

The complex ring energy $a(t)$ (with $|a|^2$ in units of energy) evolves as:

$$\frac{da}{dt} = -j\sqrt{\frac{2}{\tau_e}}\,E_i(t)
   + j\,\Delta\omega\,a(t)
   - \frac{a(t)}{\tau}$$

where $\Delta\omega = 2\pi(f_{\rm op} - f_r) + V_{\rm wr}\,V$ is the
laser–resonance detuning, and $1/\tau = 1/\tau_e + 1/\tau_l$ combines the
coupling ($\tau_e$) and loss ($\tau_l$) photon lifetimes. The transmitted
field is

$$E_o(t) = E_i(t) - j\sqrt{\frac{2}{\tau_e}}\,a(t).$$

## Mapping to the circulax DAE ($F + dQ/dt = 0$)

The `RingModulator` component below carries two internal states:

| State | $F$-term (residual) | $Q$-term (storage) | Role |
|-------|---------------------|---------------------|------|
| `a`   | $j\sqrt{2/\tau_e}\,V_{p1} - j\Delta\omega\,a + a/\tau$ | $a$ | Ring energy ODE |
| `i_out` | $V_{p2}-(V_{p1}-j\sqrt{2/\tau_e}\,a)$ | — | Output field constraint |

Port `p1` (input) contributes zero current — the ring is transparent at the
input (S11 = 0), so the source drives the input field directly. Port `p2`
(output) is driven by the auxiliary state `i_out`, which enforces the output
field relation algebraically.

The ring energy can be recovered as a post-processing step directly from the
port voltages: $a = -j\,(E_i - E_o)/\sqrt{2/\tau_e}$.

---
*References: [Absil et al., OE 2000](http://photonics.intec.ugent.be/download/pub_2992.pdf);
[Choi et al., OE 2015](https://opg.optica.org/DirectPDFAccess/D179E5F2-51D2-4CA2-AB8938DF5E6472DF_314277/oe-23-7-8762.pdf)*

In [ ]:
import diffrax
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from circulax.compiler import compile_netlist
from circulax.components.base_component import PhysicsReturn, Signals, States, component, source
from circulax.components.electronic import Resistor
from circulax.solvers import analyze_circuit, setup_transient

jax.config.update("jax_enable_x64", True)

In [ ]:
plt.rcParams.update({"axes.grid": True})
plt.style.use("dark_background")

## Component definitions

In [ ]:
@source(ports=("p1", "p2"), states=("i_src",))
def OpticalSourcePulseOnOff(
    signals: Signals,
    s: States,
    t: float,
    power: float = 1.0,
    phase: float = 0.0,
    t_on: float = 50e-12,
    t_off: float = 150e-12,
    rise: float = 5e-12,
) -> PhysicsReturn:
    """CW optical source with a smooth rectangular pulse envelope.

    Two sigmoid ramps (one up, one down) define the on/off transitions.
    The rise-time constant ``rise`` should be chosen small relative to the
    photon lifetime to approximate a sharp turn-on/off.

    Args:
        signals: Field amplitudes at positive (``p1``) and negative (``p2``) ports.
        s: Source current state variable ``i_src``.
        t: Simulation time.
        power: Peak optical power in watts. Defaults to ``1.0``.
        phase: Output field phase in radians. Defaults to ``0.0``.
        t_on: Turn-on time in seconds. Defaults to ``50e-12``.
        t_off: Turn-off time in seconds. Defaults to ``150e-12``.
        rise: Sigmoid rise/fall time constant. Defaults to ``5e-12``.

    """
    sigmoid_on = jax.nn.sigmoid((t - t_on) / rise)
    sigmoid_off = jax.nn.sigmoid((t - t_off) / rise)
    amp = jnp.sqrt(power) * jnp.exp(1j * phase) * (sigmoid_on - sigmoid_off)
    constraint = (signals.p1 - signals.p2) - amp
    return {"p1": s.i_src, "p2": -s.i_src, "i_src": constraint}, {}


@component(ports=("p1", "p2"), states=("a", "i_out"))
def RingModulator(
    signals: Signals,
    s: States,
    ng: float = 3.8,
    L: float = 3.14159265e-5,
    gamma: float = 0.976,
    alpha0: float = 0.969,
    alpha1: float = 0.0,
    voltage: float = 0.0,
    f_operating: float = 2.2904e14,
    f_resonance: float = 2.2901e14,
    v_to_wr: float = 0.0,
) -> PhysicsReturn:
    """Optical ring modulator via Temporal Coupled-Mode Theory (t-CMT).

    Tracks the complex ring energy amplitude ``a(t)`` as an internal state and
    enforces the output field relation ``E_o = E_i - j sqrt(2/tau_e) a`` via an
    algebraic constraint state ``i_out``.

    The ring energy ODE is expressed in the circulax DAE form F + dQ/dt = 0:

    - F["a"] = -(da/dt RHS) = j sqrt(2/tau_e) V_p1 - j Delta_omega a + a/tau
    - Q["a"] = a

    Gives: da/dt = -j sqrt(2/tau_e) E_i + j Delta_omega a - a/tau.

    Port ``p1`` contributes zero current (S11 = 0 approximation).
    Port ``p2`` is driven by ``i_out`` which enforces the output field.

    Args:
        signals: Complex field amplitudes at input (``p1``) and output (``p2``).
        s: Internal states: ``a`` (complex ring energy) and ``i_out`` (output
            constraint current).
        ng: Group refractive index. Defaults to ``3.8``.
        L: Ring circumference in metres. Defaults to ``2*pi*5e-6``.
        gamma: Through-coupling amplitude coefficient. Defaults to ``0.976``.
        alpha0: Voltage-independent round-trip amplitude. Defaults to ``0.969``.
        alpha1: Linear voltage coefficient of round-trip amplitude (1/V).
            Defaults to ``0.0``.
        voltage: Applied DC bias voltage. Defaults to ``0.0``.
        f_operating: Laser frequency in Hz. Defaults to ``2.2904e14``.
        f_resonance: Ring resonance frequency in Hz. Defaults to ``2.2901e14``.
        v_to_wr: Electro-optic coefficient mapping voltage to resonance shift
            in rad/s/V. Defaults to ``0.0``.

    """
    c = 2.998e8  # speed of light, m/s

    # Photon lifetimes
    tau_e = 2.0 * ng * L / ((1.0 - gamma ** 2) * c)
    alpha_v = alpha0 + alpha1 * voltage
    tau_l = 2.0 * ng * L / ((1.0 - alpha_v ** 2) * c)
    tau = 1.0 / (1.0 / tau_e + 1.0 / tau_l)

    coupling = jnp.sqrt(2.0 / tau_e)

    # Laser-resonance detuning (+ EO shift)
    delta_omega = 2.0 * jnp.pi * (f_operating - f_resonance) + v_to_wr * voltage

    # Ring energy ODE RHS: da/dt = -j*coupling*E_i + j*delta_omega*a - a/tau
    rhs_a = (-1j * coupling * signals.p1 + 1j * delta_omega * s.a - s.a / tau)

    # Output field from the ring: E_o = E_i - j*coupling*a
    E_o_expected = signals.p1 - 1j * coupling * s.a

    f = {
        "p1": 0.0 + 0.0j,               # ring is transparent at input (S11=0)
        "p2": s.i_out,
        "i_out": signals.p2 - E_o_expected,  # enforce E_o = E_i - j*coupling*a
        "a": -rhs_a,                     # ring energy ODE (negated RHS)
    }
    q = {"a": s.a}
    return f, q

## Simulation parameters

Parameters are taken from the reference silicon photonics ring modulator
in [Choi et al., OE 2015](https://opg.optica.org/DirectPDFAccess/D179E5F2-51D2-4CA2-AB8938DF5E6472DF_314277/oe-23-7-8762.pdf).

In [ ]:
c = 2.998e8  # m/s

# Ring geometry and coupling
ng = 3.8
radius_um = 5.0                          # ring radius, μm
L_ring = float(2.0 * np.pi * radius_um * 1e-6)  # circumference, m
gamma = 0.976                            # through-coupling amplitude
alpha0 = 0.969                           # round-trip amplitude (unbiased)

# Operating wavelength – 0.30 nm blue-detuned from resonance.
# This gives Δω·τ ≈ 2.4, producing ~1–2 visible ringing cycles during the
# ~3τ transient when the source turns on with a sharp (1 ps) rise time.
wavelength_res_nm = 1310.0
detuning_nm = 0.30
wavelength_op_nm = wavelength_res_nm - detuning_nm

f_resonance = c / (wavelength_res_nm * 1e-9)
f_operating = c / (wavelength_op_nm * 1e-9)

# Derived photon lifetimes
tau_e = 2.0 * ng * L_ring / ((1.0 - gamma ** 2) * c)
tau_l = 2.0 * ng * L_ring / ((1.0 - alpha0 ** 2) * c)
tau = 1.0 / (1.0 / tau_e + 1.0 / tau_l)
coupling = np.sqrt(2.0 / tau_e)

# Ringing oscillation period: beats between laser and ring resonance
delta_f = f_operating - f_resonance
T_osc = 1.0 / delta_f  # period of output power oscillation during transient

# Steady-state transmission (analytic)
A = 2.0 * tau / tau_e
x = 2.0 * np.pi * delta_f * tau  # normalised detuning Δω·τ
T_ss = np.sqrt(((1 - A) ** 2 + x ** 2) / (1 + x ** 2))

print(f"Ring circumference : L = {L_ring * 1e6:.2f} μm  (R = {radius_um} μm)")
print(f"Coupling lifetime  : τ_e = {tau_e * 1e12:.1f} ps")
print(f"Loss lifetime      : τ_l = {tau_l * 1e12:.1f} ps")
print(f"Photon lifetime    : τ   = {tau * 1e12:.2f} ps")
print(f"Laser detuning     : Δf  = {delta_f / 1e9:.1f} GHz  ({detuning_nm} nm)  →  Δω·τ = {x:.2f}")
print(f"Ringing period     : T_osc = {T_osc * 1e12:.1f} ps  (~{3 * tau / T_osc:.1f} cycles during rise)")
print(f"Steady-state |T|   : {T_ss:.3f}  ({20 * np.log10(T_ss):.1f} dB)")

## Netlist and compilation

In [ ]:
rise_time = 1e-12  # 1 ps rise/fall — sharp enough to excite ring ringing clearly

models_map = {
    "ground": lambda: 0,
    "source_pulse_onoff": OpticalSourcePulseOnOff,
    "ring_modulator": RingModulator,
    "resistor": Resistor,
}

net_dict = {
    "instances": {
        "GND": {"component": "ground"},
        "Src": {
            "component": "source_pulse_onoff",
            "settings": {
                "power": 1.0,
                "t_on": 50e-12,
                "t_off": 150e-12,
                "rise": rise_time,
            },
        },
        "Ring": {
            "component": "ring_modulator",
            "settings": {
                "ng": ng,
                "L": L_ring,
                "gamma": gamma,
                "alpha0": alpha0,
                "f_operating": float(f_operating),
                "f_resonance": float(f_resonance),
            },
        },
        "Load": {"component": "resistor", "settings": {"R": 1.0}},
    },
    "connections": {
        "GND,p1": ("Src,p2", "Load,p2"),  # common ground
        "Src,p1": "Ring,p1",              # source output → ring input
        "Ring,p2": "Load,p1",             # ring output → load
    },
}

print("Compiling netlist...")
groups, sys_size, port_map = compile_netlist(net_dict, models_map)
print(f"  sys_size = {sys_size}  (complex: {sys_size * 2} floats)")
print(f"  groups   = {list(groups.keys())}")
print(f"  port_map = {port_map}")

## DC operating point

At $t = 0$ the source is off, so the DC operating point is trivially zero.

In [ ]:
linear_strat = analyze_circuit(groups, sys_size, is_complex=True)

y_guess = jnp.zeros(sys_size * 2, dtype=jnp.float64)
y0 = linear_strat.solve_dc(groups, y_guess)
print(f"DC norm: {jnp.linalg.norm(y0):.2e}  (expected ~0 since source is off at t=0)")

## Transient simulation

The simulation runs for 250 ps, long enough to see both the ring energy
buildup ($\sim 3\tau \approx 22\,\text{ps}$ to reach 95\%) and the decay
after the optical pulse is switched off.

### Convergence note: `dtmax` in the step-size controller

With a 1 ps rise time, the adaptive PID step-size controller will eventually
grow the step to $\gg 1\,\text{ps}$. When a single step spans the entire
transition, the linear predictor lands far from the true solution, the Newton
residual becomes large, and the built-in damping (`DAMPING_FACTOR = 0.5`)
shrinks each Newton update to $\lesssim 0.5 / \Delta_{\rm max}$. With only
20 Newton iterations this is not enough to converge, regardless of `dt0`.

The fix is **`dtmax = rise_time / 2`** in `PIDController`, which prevents any
single step from spanning the source transition. This keeps the predictor
accurate and Newton convergence fast.

In [ ]:
t_end = 250e-12  # 250 ps
num_points = 2500

transient_sim = setup_transient(groups=groups, linear_strategy=linear_strat)

controller = diffrax.PIDController(
    rtol=1e-4,
    atol=1e-6,
    dtmax=rise_time / 2,   # keep steps within the source rise time
)

print("Running transient simulation...")
sol = transient_sim(
    t0=0.0,
    t1=t_end,
    dt0=1e-13,
    y0=y0,
    saveat=diffrax.SaveAt(ts=jnp.linspace(0.0, t_end, num_points)),
    max_steps=500000,
    throw=False,
    stepsize_controller=controller,
)

if sol.result == diffrax.RESULTS.successful:
    print("Simulation successful")
else:
    print(f"Simulation ended with: {sol.result}")

## Results

The ring energy $a(t)$ is read directly from the solution vector via
`port_map["Ring,a"]`, alongside the input and output fields.

In [ ]:
ys_complex = sol.ys[:, :sys_size] + 1j * sol.ys[:, sys_size:]
ts_ps = sol.ts * 1e12

E_in = ys_complex[:, port_map["Src,p1"]]
E_out = ys_complex[:, port_map["Ring,p2"]]
a_val = ys_complex[:, port_map["Ring,a"]]

P_in = jnp.abs(E_in) ** 2
P_out = jnp.abs(E_out) ** 2
eps = 1e-20

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

t_on_ps = 50.0
T_osc_ps = T_osc * 1e12

axes[0].plot(ts_ps, jnp.abs(E_in), color="tab:blue", linewidth=1.5, label="|E_in| (source)")
axes[0].plot(ts_ps, jnp.abs(E_out), color="tab:orange", linewidth=1.5, label="|E_out| (transmitted)")
axes[0].axhline(T_ss, color="tab:orange", linestyle="--", linewidth=0.8,
                label=f"Steady-state |T| = {T_ss:.3f}")
axes[0].set_ylabel("|E| (a.u.)")
axes[0].set_title(
    f"Optical Impulse Response — ring modulator  "
    f"(Δλ = {detuning_nm} nm,  Δω·τ = {x:.1f},  T_osc = {T_osc_ps:.1f} ps)"
)
axes[0].legend(loc="upper right")

axes[1].plot(ts_ps, jnp.abs(a_val) ** 2, color="tab:green", linewidth=1.5)
axes[1].set_ylabel("|a|² (ring energy, a.u.)")
axes[1].set_title(f"Ring energy  (τ ≈ {tau * 1e12:.1f} ps,  ringing period ≈ {T_osc_ps:.1f} ps)")

a2_ss = (2.0 / tau_e) * tau ** 2 / (1 + x ** 2)
axes[1].annotate(
    f"T_osc ≈ {T_osc_ps:.0f} ps",
    xy=(t_on_ps + T_osc_ps, a2_ss * 0.5),
    xytext=(t_on_ps + T_osc_ps + 8, a2_ss * 0.25),
    arrowprops=dict(arrowstyle="->", color="white"),
    color="white",
    fontsize=9,
)

P_out_norm = P_out / (P_in + eps)
axes[2].plot(ts_ps, P_out_norm, color="tab:red", linewidth=1.5)
axes[2].axhline(T_ss ** 2, color="tab:red", linestyle="--", linewidth=0.8,
                label=f"Steady-state |T|² = {T_ss**2:.3f}")
axes[2].axvspan(t_on_ps, t_on_ps + 3 * tau * 1e12, alpha=0.08, color="white",
                label=f"~3τ transient ({3 * tau * 1e12:.0f} ps)")
axes[2].set_ylabel("|E_out|² / |E_in|²")
axes[2].set_xlabel("Time (ps)")
axes[2].set_title("Normalised power transmission — ringing at turn-on edge")
axes[2].legend(loc="upper right")
axes[2].set_ylim(bottom=0)

for ax in axes:
    ax.axvline(50, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)
    ax.axvline(150, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)

fig.text(0.5, 0.01, "Gray dotted lines mark t_on = 50 ps and t_off = 150 ps",
         ha="center", color="gray", fontsize=8)

fig.tight_layout()
plt.show()

mid = num_points // 2
print(f"Simulated steady-state |T|: {float(jnp.abs(E_out[mid]) / (jnp.abs(E_in[mid]) + eps)):.3f}")
print(f"Analytic  steady-state |T|: {T_ss:.3f}")

## Small-signal electro-optic bandwidth

> **Note — time-domain is overkill here.**
> A proper small-signal AC analysis or a Harmonic Balance (HB) solver would
> extract the EO frequency response in a single solve at each frequency, without
> integrating through many AC periods.  circulax already has a Harmonic Balance
> solver and small-signal AC analysis is a planned feature.
> The purpose of this section is to verify that the **time-domain DAE equations
> are equivalent** to the analytic transfer function — i.e. the equations are
> correct — not to advocate transient simulation as the right tool for bandwidth
> characterisation in production.

A small sinusoidal voltage is applied to the ring modulator through an RC
equivalent circuit that models a reverse-biased PN junction:

- **R_s** — series resistance (contact + waveguide)
- **C_j** — depletion capacitance (junction)

The combined small-signal EO response is limited by two first-order poles:

$$|H(f)| \propto \frac{1}{\sqrt{1 + (f/f_{\rm RC})^2}\;\sqrt{1 + (f/f_{\rm opt})^2}}$$

where $f_{\rm RC} = 1/(2\pi R_s C_j)$ and $f_{\rm opt} = 1/(2\pi\tau)$.

**Simulation strategy**: the DC operating point (optical CW + DC bias $V_{\rm bias}$)
is found first and used as the initial condition at $t = 0$.  A small AC swing
$V_{\rm ac}\sin(2\pi f t)$ is then enabled at $t = 10\,\text{ps}$.
For each modulation frequency the output power amplitude (max $-$ min of $|E_{\rm out}|^2$)
is extracted from the last half of the simulation and normalised to the
low-frequency value.

In [ ]:
from circulax.components.electronic import Capacitor
from circulax.utils import update_group_params


@source(ports=("p1", "p2"), states=("i_src",))
def OpticalSourceStep(
    signals: Signals,
    s: States,
    t: float,
    power: float = 1.0,
    phase: float = 0.0,
) -> PhysicsReturn:
    """Constant-amplitude optical CW source (always on, no time dependence)."""
    amp = jnp.sqrt(power) * jnp.exp(1j * phase)
    constraint = (signals.p1 - signals.p2) - amp
    return {"p1": s.i_src, "p2": -s.i_src, "i_src": constraint}, {}


@source(ports=("p1", "p2"), states=("i_src",))
def BiasedACSource(
    signals: Signals,
    s: States,
    t: float,
    V_bias: float = 2.0,
    V_ac: float = 0.1,
    freq: float = 1e9,
    t_ac_start: float = 10e-12,
    rise_ac: float = 1e-12,
) -> PhysicsReturn:
    """DC-biased sinusoidal voltage source.

    Outputs V_bias (always present) plus V_ac * sin(2π * freq * t), with the
    AC component enabled smoothly at t_ac_start.  At t=0 the AC term is
    negligibly small (sigmoid(-10) ≈ 0), so solve_dc finds the correct DC
    operating point with only V_bias present.
    """
    v_ac = V_ac * jnp.sin(2.0 * jnp.pi * freq * t)
    ac_enable = jax.nn.sigmoid((t - t_ac_start) / rise_ac)
    v_total = V_bias + v_ac * ac_enable
    constraint = (signals.p1 - signals.p2) - v_total
    return {"p1": s.i_src, "p2": -s.i_src, "i_src": constraint}, {}


@component(ports=("p1", "p2", "v_e"), states=("a", "i_out"))
def RingModulatorEO(
    signals: Signals,
    s: States,
    ng: float = 3.8,
    L: float = 3.14159265e-5,
    gamma: float = 0.976,
    alpha0: float = 0.969,
    alpha1: float = 0.0,
    f_operating: float = 2.2904e14,
    f_resonance: float = 2.2901e14,
    v_to_wr: float = 0.0,
) -> PhysicsReturn:
    """Ring modulator with an external electrical voltage port ``v_e``.

    The voltage at port ``v_e`` modulates the ring resonance frequency via
    ``v_to_wr`` (rad/s/V).  The port is high-impedance: it draws no current
    (``F["v_e"] = 0``), so any driver circuit sees an open circuit.

    The physics is otherwise identical to :func:`RingModulator`, but ``voltage``
    is read from the external port rather than being a fixed parameter.
    """
    c = 2.998e8
    voltage = jnp.real(signals.v_e)  # electrical port — real part only

    tau_e = 2.0 * ng * L / ((1.0 - gamma ** 2) * c)
    alpha_v = alpha0 + alpha1 * voltage
    tau_l = 2.0 * ng * L / ((1.0 - alpha_v ** 2) * c)
    tau = 1.0 / (1.0 / tau_e + 1.0 / tau_l)
    coupling = jnp.sqrt(2.0 / tau_e)

    delta_omega = 2.0 * jnp.pi * (f_operating - f_resonance) + v_to_wr * voltage

    rhs_a = -1j * coupling * signals.p1 + 1j * delta_omega * s.a - s.a / tau
    E_o_expected = signals.p1 - 1j * coupling * s.a

    f = {
        "p1":    0.0 + 0.0j,
        "p2":    s.i_out,
        "v_e":   0.0 + 0.0j,              # high-impedance electrical port
        "i_out": signals.p2 - E_o_expected,
        "a":     -rhs_a,
    }
    q = {"a": s.a}
    return f, q

In [ ]:
# ── Electrical RC parameters ──────────────────────────────────────────────────
V_bias     = 2.0       # V   — DC reverse bias
V_ac       = 0.2       # V   — small-signal swing (<<1, so response is linear)
R_s        = 100.0     # Ω   — series resistance (contact + waveguide)
C_j        = 100e-15   # F   — junction depletion capacitance
v_to_wr    = -2.0 * np.pi * 2e9   # rad/s/V — EO coefficient (2 GHz/V resonance shift)
t_ac_start = 10e-12    # s   — delay before AC is enabled

f_RC    = 1.0 / (2.0 * np.pi * R_s * C_j)   # electrical 3-dB bandwidth
f_opt_bw = 1.0 / (2.0 * np.pi * tau)         # optical photon-lifetime bandwidth

print(f"Electrical RC bandwidth : f_RC  = {f_RC  / 1e9:.1f} GHz")
print(f"Optical photon-lifetime : f_opt = {f_opt_bw / 1e9:.1f} GHz")
print(f"EO coefficient          : dω/dV = {v_to_wr / (2e9 * np.pi):.1f} × 2π GHz/V")

# ── Models map ────────────────────────────────────────────────────────────────
models_map_ss = {
    "ground":      lambda: 0,
    "optical_cw":  OpticalSourceStep,
    "biased_ac":   BiasedACSource,
    "ring_eo":     RingModulatorEO,
    "resistor":    Resistor,
    "capacitor":   Capacitor,
}

# ── Netlist ───────────────────────────────────────────────────────────────────
# Topology:
#   OptSrc → Ring (optical path)
#   Vsrc → Rs → node_ve → Cj → GND  (electrical RC network)
#   Ring,v_e = node_ve  (high-impedance; ring reads voltage, draws no current)
net_dict_ss = {
    "instances": {
        "GND":    {"component": "ground"},
        "OptSrc": {"component": "optical_cw",  "settings": {"power": 1.0}},
        "Ring":   {"component": "ring_eo",     "settings": {
            "ng": ng, "L": L_ring, "gamma": gamma, "alpha0": alpha0,
            "f_operating": float(f_operating), "f_resonance": float(f_resonance),
            "v_to_wr": v_to_wr,
        }},
        "Load":   {"component": "resistor",    "settings": {"R": 1.0}},
        "Vsrc":   {"component": "biased_ac",   "settings": {
            "V_bias": V_bias, "V_ac": V_ac, "freq": 1e9, "t_ac_start": t_ac_start,
        }},
        "Rs":     {"component": "resistor",    "settings": {"R": R_s}},
        "Cj":     {"component": "capacitor",   "settings": {"C": C_j}},
    },
    "connections": {
        "GND,p1":   ("OptSrc,p2", "Load,p2", "Vsrc,p2", "Cj,p2"),
        "OptSrc,p1": "Ring,p1",
        "Ring,p2":  "Load,p1",
        "Vsrc,p1":  "Rs,p1",
        "Rs,p2":    ("Cj,p1", "Ring,v_e"),   # node_ve: RC junction & ring voltage port
    },
}

print("\nCompiling small-signal netlist...")
groups_ss, sys_size_ss, port_map_ss = compile_netlist(net_dict_ss, models_map_ss)
print(f"  sys_size = {sys_size_ss}  (complex: {sys_size_ss * 2} floats)")
print(f"  groups   = {list(groups_ss.keys())}")
print(f"  port_map = {port_map_ss}")

# ── DC operating point ────────────────────────────────────────────────────────
# At t=0: BiasedACSource outputs V_bias (AC term is sigmoid(-10) ≈ 0).
# OpticalSourceStep is always on.  solve_dc finds the joint optical + electrical
# steady state: ring at optical SS, node_ve = V_bias.
linear_strat_ss = analyze_circuit(groups_ss, sys_size_ss, is_complex=True)
y_guess_ss = jnp.zeros(sys_size_ss * 2, dtype=jnp.float64)
y0_ss = linear_strat_ss.solve_dc(groups_ss, y_guess_ss)

# Report DC state
ys0_complex = y0_ss[:sys_size_ss] + 1j * y0_ss[sys_size_ss:]
V_ve_dc = float(jnp.real(ys0_complex[port_map_ss["Ring,v_e"]]))
E_in_dc  = ys0_complex[port_map_ss["Ring,p1"]]
E_out_dc = ys0_complex[port_map_ss["Ring,p2"]]
T_dc_sim = float(jnp.abs(E_out_dc) / (jnp.abs(E_in_dc) + 1e-20))

# Analytic DC transmission at the biased detuning
delta_omega_dc = 2.0 * np.pi * (f_operating - f_resonance) + v_to_wr * V_bias
x_dc  = delta_omega_dc * tau
A_dc  = 2.0 * tau / tau_e
T_dc_analytic = np.sqrt(((1 - A_dc) ** 2 + x_dc ** 2) / (1 + x_dc ** 2))

print(f"\nDC node_ve voltage : {V_ve_dc:.3f} V  (expected {V_bias:.1f} V)")
print(f"DC |T| simulated   : {T_dc_sim:.4f}")
print(f"DC |T| analytic    : {T_dc_analytic:.4f}")
print(f"DC Δω·τ            : {x_dc:.3f}")

In [ ]:
freqs_GHz  = np.linspace(1.0, 80, 10)
N_periods  = 10.0      # simulate this many AC periods per run
n_save     = 500       # save this many points in the readout window
amplitudes = []        # will hold max(P_out) - min(P_out) per frequency

transient_sim_ss = setup_transient(groups=groups_ss, linear_strategy=linear_strat_ss)
E_out_node = port_map_ss["Ring,p2"]

print(f"Running {len(freqs_GHz)}-point frequency sweep …")
for idx, f_ghz in enumerate(freqs_GHz):
    f_hz = f_ghz * 1e9

    groups_f = update_group_params(groups_ss, "biased_ac", "freq", f_hz)

    t_sim = t_ac_start + N_periods / f_hz
    t_read = t_ac_start + (N_periods // 2) / f_hz
    ts_save = jnp.linspace(t_read, t_sim, n_save)

    dtmax = 1.0 / (20.0 * f_hz)   # ≥ 20 samples per AC period

    sol = transient_sim_ss(
        t0=0.0,
        t1=t_sim,
        dt0=min(1e-12, dtmax),
        y0=y0_ss,
        saveat=diffrax.SaveAt(ts=ts_save),
        max_steps=2_000_000,
        throw=False,
        args=(groups_f, sys_size_ss),
        stepsize_controller=diffrax.PIDController(rtol=1e-8, atol=1e-8, dtmax=dtmax),
    )

    if sol.result != diffrax.RESULTS.successful:
        print(f"  [{idx+1:2d}/{len(freqs_GHz)}] {f_ghz:5.1f} GHz  WARNING: {sol.result}")

    ys_c   = sol.ys[:, :sys_size_ss] + 1j * sol.ys[:, sys_size_ss:]
    P_out  = jnp.abs(ys_c[:, E_out_node]) ** 2
    P_norm = jnp.min(P_out[:int(-n_save/2)])
    amplitudes.append(float(jnp.max(P_out[:int(-n_save/2)])-P_norm))

    if (idx + 1) % 2 == 0:
        print(f"  {idx + 1}/{len(freqs_GHz)} done")

print("Sweep complete.")

In [ ]:
omega_m = 2.0 * np.pi * freqs_GHz * 1e9   # rad/s

delta_omega_dc = (
    2.0 * np.pi * (float(f_operating) - float(f_resonance))
    + v_to_wr * V_bias
)
inv_tau   = 1.0 / tau
inv_tau_l = 1.0 / tau_l

# Electrical RC lowpass
H_RC = 1.0 / (1.0 + 1j * omega_m * R_s * C_j)

# Second-order optical transfer function
#   H_opt(jω) = (jω + 2/τ_l) / (Δω² + (1/τ)² − ω² + j(2/τ)ω)
denom_dc    = delta_omega_dc ** 2 + inv_tau ** 2
H_opt       = (1j * omega_m + 2.0 * inv_tau_l) / (
                  -omega_m ** 2 + 1j * 2.0 * inv_tau * omega_m + denom_dc
              )

H_total     = H_RC * H_opt
H_mag       = np.abs(H_total)
H_norm      = H_mag / H_mag[0]
H_dB        = 20.0 * np.log10(H_norm)

H_RC_dB  = 20.0 * np.log10(np.abs(H_RC)  / np.abs(H_RC[0]))
H_opt_dB = 20.0 * np.log10(np.abs(H_opt) / np.abs(H_opt[0]))

f_pole_GHz = np.sqrt(delta_omega_dc ** 2 + inv_tau ** 2) / (2.0 * np.pi * 1e9)

amps      = np.array(amplitudes)
amps_norm = amps / (amps[0])
amps_dB   = 20.0 * np.log10(amps_norm)

print(f"DC-biased detuning  Δf      : {delta_omega_dc / (2*np.pi*1e9):.1f} GHz  (Δω·τ = {delta_omega_dc * tau:.2f})")
print(f"Optical pole magnitude      : {f_pole_GHz:.1f} GHz")
print(f"Optical peak (no RC)        : {float(np.max(H_opt_dB)):.1f} dB  at {freqs_GHz[np.argmax(H_opt_dB)]:.1f} GHz")
print(f"Combined peak (RC + optical): {float(np.max(H_dB)):.1f} dB  at {freqs_GHz[np.argmax(H_dB)]:.1f} GHz")

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(freqs_GHz, amps_dB, "o-",  ms=7.5, linewidth=2.0, zorder=3,
            label="Transient (max − min of $|E_{out}|^2$)")
ax.plot(freqs_GHz, H_dB,    "w--",       linewidth=3.0,
            label="Analytic: RC × optical (combined)")
ax.plot(freqs_GHz, H_opt_dB, ":",  linewidth=2.0, alpha=0.6,
            label=r"Optical alone: $(j\omega\!+\!2/\tau_l)/(…)$  [peaks near $|\Delta\omega|$]")
ax.plot(freqs_GHz, H_RC_dB,  ":",  linewidth=2.0, alpha=0.6,
            label=f"RC alone: $1/(1+j\\omega R_s C_j)$  [$f_{{RC}}$={f_RC/1e9:.0f} GHz]")

ax.axhline(-3.0, color="gray", linestyle=":", linewidth=0.9, label="−3 dB")
ax.axvline(abs(delta_omega_dc) / (2*np.pi*1e9), color="tab:orange",
           linestyle=":", linewidth=0.9, alpha=0.7,
           label=f"|Δf| = {abs(delta_omega_dc)/(2*np.pi*1e9):.0f} GHz (optical peak region)")

ax.set_xlabel("Modulation frequency (GHz)")
ax.set_ylabel("Normalised EO response (dB)")
ax.set_title(
    f"Small-signal EO response — RC × optical  "
    f"($V_{{bias}}$={V_bias:.0f} V, $\\Delta\\omega\\cdot\\tau$={delta_omega_dc*tau:.1f}, "
    f"$R_s$={R_s:.0f} Ω, $C_j$={C_j*1e15:.0f} fF)"
)
ax.set_xlim(freqs_GHz[0], freqs_GHz[-1])
ax.set_ylim(-15.0, 12.0)
ax.legend(fontsize=8, loc="lower left")
fig.tight_layout()
plt.show()

---
## Part 3: EO Bandwidth via Harmonic Balance

The transient sweep above runs 25 independent simulations — one per frequency.
**Harmonic Balance (HB)** finds the periodic steady state *directly*, without
time-stepping to steady state, and `jax.vmap` solves all frequencies in a
single XLA compilation.

The modulation frequency $f_m$ is the HB fundamental.  With $N_h = 5$ harmonics
($K = 11$ time samples per period) the Newton loop converges in ~20 iterations
independent of $f_m$.

### How the sweep works

```python
def hb_sweep_point(freq):             # freq is now a JAX argument — vmappable
    run_hb = setup_harmonic_balance(groups_hb, num_vars_hb,
                                    freq=freq, num_harmonics=N_harm_hb,
                                    is_complex=True)   # photonic circuit
    y_time, _ = run_hb(y_dc_hb)
    E_out = y_time[:, vout_hb] + 1j * y_time[:, vout_hb + num_vars_hb]
    P_out = jnp.abs(E_out) ** 2
    return jnp.max(P_out) - jnp.min(P_out)

amps_hb = jax.jit(jax.vmap(hb_sweep_point))(sweep_freqs)
```

`is_complex=True` tells the HB solver to use the unrolled `[re | im]` block
representation that photonic circuits require.

In [ ]:
from circulax import setup_harmonic_balance


@source(ports=("p1", "p2"), states=("i_src",))
def BiasedSinSource(
    signals: Signals,
    s: States,
    t: float,
    V_bias: float = 2.0,
    V_ac: float = 0.1,
    freq: float = 1e9,
) -> PhysicsReturn:
    """DC-biased sinusoidal source, exactly periodic at ``freq``.

    At t = 0: V = V_bias (no AC), giving the same DC operating point as
    :class:`BiasedACSource` with the AC component disabled.
    """
    v = V_bias + V_ac * jnp.sin(2.0 * jnp.pi * freq * t)
    return {"p1": s.i_src, "p2": -s.i_src, "i_src": (signals.p1 - signals.p2) - v}, {}


models_map_hb = {
    "ground":      lambda: 0,
    "optical_cw":  OpticalSourceStep,
    "biased_sin":  BiasedSinSource,
    "ring_eo":     RingModulatorEO,
    "resistor":    Resistor,
    "capacitor":   Capacitor,
}

net_dict_hb = {
    "instances": {
        "GND":    {"component": "ground"},
        "OptSrc": {"component": "optical_cw",  "settings": {"power": 1.0}},
        "Ring":   {"component": "ring_eo",     "settings": {
            "ng": ng, "L": L_ring, "gamma": gamma, "alpha0": alpha0,
            "f_operating": float(f_operating), "f_resonance": float(f_resonance),
            "v_to_wr": v_to_wr,
        }},
        "Load":   {"component": "resistor",    "settings": {"R": 1.0}},
        "Vsrc":   {"component": "biased_sin",  "settings": {
            "V_bias": V_bias, "V_ac": V_ac, "freq": 1e9,   # placeholder; updated per sweep point
        }},
        "Rs":     {"component": "resistor",    "settings": {"R": R_s}},
        "Cj":     {"component": "capacitor",   "settings": {"C": C_j}},
    },
    "connections": {
        "GND,p1":    ("OptSrc,p2", "Load,p2", "Vsrc,p2", "Cj,p2"),
        "OptSrc,p1": "Ring,p1",
        "Ring,p2":   "Load,p1",
        "Vsrc,p1":   "Rs,p1",
        "Rs,p2":     ("Cj,p1", "Ring,v_e"),
    },
}

print("Compiling HB netlist...")
groups_hb, num_vars_hb, port_map_hb = compile_netlist(net_dict_hb, models_map_hb)
print(f"  num_vars = {num_vars_hb}  (unrolled: {num_vars_hb * 2} real floats)")
print(f"  port_map = {port_map_hb}")

linear_strat_hb = analyze_circuit(groups_hb, num_vars_hb, is_complex=True)
y_dc_hb = linear_strat_hb.solve_dc(groups_hb, jnp.zeros(num_vars_hb * 2, dtype=jnp.float64))

ys0_hb = y_dc_hb[:num_vars_hb] + 1j * y_dc_hb[num_vars_hb:]
E_in_dc_hb  = ys0_hb[port_map_hb["Ring,p1"]]
E_out_dc_hb = ys0_hb[port_map_hb["Ring,p2"]]
T_dc_hb = float(jnp.abs(E_out_dc_hb) / (jnp.abs(E_in_dc_hb) + 1e-20))
print(f"\nDC |T| (HB netlist) : {T_dc_hb:.4f}  (transient: {T_dc_sim:.4f})")

In [ ]:
N_harm_hb = 5   # 5 harmonics → K = 11 time samples per period
K_hb      = 2 * N_harm_hb + 1
f_demo    = 1e9  # 1 GHz single-point demo

groups_hb_demo = update_group_params(groups_hb, "biased_sin", "freq", f_demo)

run_hb_demo = setup_harmonic_balance(
    groups_hb_demo, num_vars_hb, freq=f_demo, num_harmonics=N_harm_hb, is_complex=True
)
y_time_demo, _ = run_hb_demo(y_dc_hb)
print(f"HB converged at {f_demo/1e9:.0f} GHz.  y_time shape: {y_time_demo.shape}")

# Reconstruct complex optical fields from the unrolled [re | im] state.
vout_hb = port_map_hb["Ring,p2"]
vin_hb  = port_map_hb["Ring,p1"]
T_period_demo = 1.0 / f_demo
t_hb_demo = np.linspace(0, T_period_demo * 1e9, K_hb, endpoint=False)  # ns

E_in_demo  = y_time_demo[:, vin_hb]  + 1j * y_time_demo[:, vin_hb  + num_vars_hb]
E_out_demo = y_time_demo[:, vout_hb] + 1j * y_time_demo[:, vout_hb + num_vars_hb]
P_in_demo  = jnp.abs(E_in_demo)  ** 2
P_out_demo = jnp.abs(E_out_demo) ** 2

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(t_hb_demo, np.array(P_in_demo),  "C0o-", ms=7, label=r"$|E_\mathrm{in}|^2$  (CW)")
ax.plot(t_hb_demo, np.array(P_out_demo), "C1s-", ms=7, label=r"$|E_\mathrm{out}|^2$  (modulated)")
ax.axhline(T_dc_hb ** 2, color="C1", ls="--", lw=0.8, label=f"DC $|T|^2 = {T_dc_hb**2:.3f}$")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Power (a.u.)")
ax.set_title(
    f"HB waveform at $f_m$ = {f_demo/1e9:.0f} GHz  "
    f"($K$ = {K_hb} points, $N_h$ = {N_harm_hb})"
)
ax.legend()
plt.tight_layout()
plt.show()

amp_demo = float(jnp.max(P_out_demo) - jnp.min(P_out_demo))
print(f"P_out oscillation amplitude: {amp_demo:.4e} W  (peak-to-peak power modulation)")

In [ ]:
def hb_sweep_point(freq):
    """HB solve at one modulation frequency — vmappable over freq."""
    g_f = update_group_params(groups_hb, "biased_sin", "freq", freq)

    run_hb = setup_harmonic_balance(
        g_f, num_vars_hb, freq=freq, num_harmonics=N_harm_hb, is_complex=True
    )
    y_time, _ = run_hb(y_dc_hb)

    E_out = y_time[:, vout_hb] + 1j * y_time[:, vout_hb + num_vars_hb]
    P_out = jnp.abs(E_out) ** 2
    return jnp.max(P_out) - jnp.min(P_out)


freqs_GHz_HB = jnp.array(freqs_GHz-0.5*jnp.diff(freqs_GHz)[0]) #offsetting only for visualization
sweep_freqs_hb = freqs_GHz_HB*1e9

print(f"Running vmapped HB sweep over {len(sweep_freqs_hb)} frequencies (1–50 GHz)...")
amps_hb = jax.jit(jax.vmap(hb_sweep_point))(sweep_freqs_hb)
print("Done.")

amps_hb_np   = np.array(amps_hb)
amps_hb_norm = amps_hb_np / (amps_hb_np[0] + 1e-30)
amps_hb_dB   = 20.0 * np.log10(amps_hb_norm + 1e-30)

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    freqs_GHz, amps_dB, "o-", ms=7.5, linewidth=2.0, zorder=3,
    label="Transient",
)
ax.plot(
    freqs_GHz_HB, amps_hb_dB, "D--", ms=5, linewidth=1.5, zorder=4,
    label=f"HB vmap ({len(sweep_freqs_hb)} freqs, one XLA call, $N_h$={N_harm_hb})",
)
ax.plot(freqs_GHz, H_dB, "w-", linewidth=2.0, alpha=0.7,
            label="Analytic: RC × optical")
ax.plot(freqs_GHz, H_opt_dB, ":", linewidth=2.0, alpha=0.5,
            label=r"Optical alone: $(j\omega + 2/\tau_l)/(…)$")
ax.plot(freqs_GHz, H_RC_dB,  ":", linewidth=2.0, alpha=0.5,
            label=f"RC alone  [$f_{{RC}}$ = {f_RC/1e9:.0f} GHz]")

ax.axhline(-3.0, color="gray", linestyle=":", linewidth=1.5, label="−3 dB")
ax.axvline(
    abs(delta_omega_dc) / (2 * np.pi * 1e9),
    color="tab:orange", linestyle=":", linewidth=0.9, alpha=0.7,
    label=f"|Δf| = {abs(delta_omega_dc)/(2*np.pi*1e9):.0f} GHz",
)

ax.set_xlabel("Modulation frequency (GHz)")
ax.set_ylabel("Normalised EO response (dB)")
ax.set_title(
    f"EO bandwidth — HB vmap vs transient vs analytic  "
    f"($V_{{bias}}$={V_bias:.0f} V, $\\Delta\\omega\\cdot\\tau$={delta_omega_dc*tau:.1f})"
)
ax.set_xlim(freqs_GHz[0], freqs_GHz[-1])
ax.set_ylim(-15.0, 12.0)
ax.legend(fontsize=8, loc="lower left")
fig.tight_layout()
plt.show()

print("\nFreq (GHz)  | Transient (dB) | HB vmap (dB) | Analytic (dB)")
print("-" * 60)
for f, a_tr, a_hb, a_an in zip(freqs_GHz[::5], amps_dB[::5], amps_hb_dB[::5], H_dB[::5]):
    print(f"  {f:5.1f}       |   {a_tr:+6.2f}        |  {a_hb:+6.2f}       | {a_an:+6.2f}")